<a href="https://colab.research.google.com/github/jhuangjennifer/573ChineseEnglishSummarization/blob/jjnhuang-work/notebooks/models/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from transformers import BartTokenizer, BartModel
import torch

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

model = BartModel.from_pretrained(
    'facebook/bart-large',
    force_download=True
)

inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
print("model shape:", last_hidden_states.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model shape: torch.Size([1, 8, 1024])


In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import Dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

print("library load")

library load


In [4]:
pip install transformers

In [5]:
# dataset check
DATASET_NAME = "XSAMSum"   # you can change this "XMediaSum40k"
MODEL_NAME = "facebook/bart-large"
CACHE_DIR = "./hf_cache"
BASE_DIR = f"sample_data"

In [6]:
# dataset load
import os
from datasets import load_dataset

#BASE_DIR = f"../../data/raw/{DATASET_NAME}"

data_files = {
    "train": os.path.join(BASE_DIR, "train.json"),
    "validation": os.path.join(BASE_DIR, "val.json"),
    "test": os.path.join(BASE_DIR, "test.json"),
}

print(data_files)

dataset_dict = load_dataset("json", data_files=data_files)
dataset_dict

{'train': 'sample_data/train.json', 'validation': 'sample_data/val.json', 'test': 'sample_data/test.json'}


DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 818
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 819
    })
})

In [12]:
import pandas as pd

# DataFrame
df_train = dataset_dict['train'].to_pandas()

# result
display(df_train.head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [13]:
import pandas as pd

# convert into DataFrame
df_train = dataset_dict['train'].to_pandas()
df_val = dataset_dict['validation'].to_pandas()
df_test = dataset_dict['test'].to_pandas()

# result check
print(f"Train rows: {len(df_train)}")
print(f"Validation rows: {len(df_val)}")
print(f"Test rows: {len(df_test)}")

Train rows: 14732
Validation rows: 818
Test rows: 819


In [14]:
display(df_train.head())
display(df_val.head())
display(df_test.head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


,dialogue,summary,summary_de,summary_zh
0,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...,"a wird morgen ins tierheim gehen, um einen wel...",a明天要去动物收容所给她儿子买一只小狗。他们上周一已经去了收容所，她儿子选了这只小狗。
1,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...,emma und rob lieben den adventskalender. laure...,艾玛和罗伯都很喜欢这个基督降临历。劳伦在日历内放各种物品，譬如小玩具和圣诞装饰品。她的孩子一...
2,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...,"madison ist schwanger, aber sie möchte darüber...",麦迪逊怀孕了，但她不想提及。帕特丽夏·史蒂文斯结婚了，她认为在这之前自己就怀孕了。
3,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.,marla fand unter ihrem bett ein paar unterhosen.,玛拉在床底下发现了两条四角裤。
4,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...,"robert möchte, dass fred ihm die adresse des m...",罗伯特想让弗雷德把音像店的地址发给他，因为他需要买吉他弦。


,dialogue,summary,summary_de,summary_zh
0,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...,"hannah braucht bettys nummer, aber amanda hat ...",汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。
1,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,Eric and Rob are going to watch a stand-up on ...,eric und rob werden sich ein stand-up auf yout...,埃里克和罗伯要在youtube上看一场单口相声。
2,"Lenny: Babe, can you help me with something?\r...",Lenny can't decide which trousers to buy. Bob ...,"lenny kann sich nicht entscheiden, welche hose...",莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...,emma kommt bald nach hause sein und sie wird e...,艾玛很快就会回家，而且她会告诉威尔。
4,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",Jane is in Warsaw. Ollie and Jane has a party....,jane ist in warschau. ollie und jane haben ein...,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...


In [15]:
dfs = {split: dataset.to_pandas() for split, dataset in dataset_dict.items()}

display(dfs['train'].head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [16]:
# pick random sample
sample = df_train.sample(1)
print(f"dialogue: {sample['dialogue'].values[0]}")
print(f"summary: {sample['summary'].values[0]}")
print(f"summary_de: {sample['summary_de'].values[0]}")
print(f"summary_z: {sample['summary_zh'].values[0]}")

dialogue: Stacy: what the fuck happened when I was gone?
Stacy: <file_photo>
Henry: don't know, the bathroom looked fine the last time I went there
Anthony: shiiit
Javier: I think David's girlfriend got sick 
Javier: it could be hers
Stacy: ok give me David's phone
Stacy: if this chick won't repay it I'll make David pay
Javier: that sounded really scary
Henry: how drunk you have to be to do sth like that
Javier: 999 888 777
Stacy: I don't care who did it and how
Stacy: I just want it repaired
Anthony: that's why I never invite anyone
Javier: i hope you'll have it repaired soon
Anthony: that's so lame of her
Stacy: gotta go, thx for David's nr
summary: Someone destroyed Stacy's bathroom and the main suspect is David's girlfriend. Javier gives Stacy David's phone number so he or she would pay for the repair.
summary_de: jemand hat stacys badezimmer zerstört und der hauptverdächtige ist davids freundin. javier gibt stacy davids telefonnummer, damit er oder sie für die reparatur bezahlen k

In [17]:
import pandas as pd
import re

# 1. Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return text
    text = text.replace('\r', '')          # Remove carriage returns
    text = re.sub(r'\s+', ' ', text)       # Replace multiple spaces with a single space
    return text.strip()                    # Remove leading and trailing spaces

# 2. Integrated preprocessing pipeline function
def preprocess_to_clean_df(dataset_split):
    """
    Takes a Hugging Face Dataset as input, extracts only the 'dialogue' and 'summary' columns,
    and returns a DataFrame with missing values and duplicates removed, and text normalized.
    """
    # Convert to Pandas DataFrame
    df = dataset_split.to_pandas()

    # Select only the necessary columns
    df = df[['dialogue', 'summary', 'summary_zh']].copy()

    # Drop missing values
    df = df.dropna(subset=['dialogue', 'summary', 'summary_zh'])

    # Apply text normalization
    df['dialogue'] = df['dialogue'].apply(clean_text)
    df['summary'] = df['summary'].apply(clean_text)
    df['summary_zh'] = df['summary_zh'].apply(clean_text)

    # Remove duplicate data and reset index
    df = df.drop_duplicates().reset_index(drop=True)

    return df

# 3. Create DataFrames with updated variable names
train_data = preprocess_to_clean_df(dataset_dict['train'])
val_data = preprocess_to_clean_df(dataset_dict['validation'])
test_data = preprocess_to_clean_df(dataset_dict['test'])

# 4. Check results
print("Preprocessing results!")
print(f"train_data: {train_data.shape}")
print(f"val_data: {val_data.shape}")
print(f"test_data: {test_data.shape}\n")

# Check a sample
display(train_data.head())

Preprocessing results!
train_data: (14732, 3)
val_data: (818, 3)
test_data: (819, 3)



,dialogue,summary,summary_zh
0,Amanda: I baked cookies. Do you want some? Jer...,Amanda baked cookies and will bring Jerry some...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up? Kim: Bad mood tbh, I was g...",Kim may try the pomodoro technique recommended...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something Sam: i d...,"Sam is confused, because he overheard Rick com...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [18]:
train_data['dialogue'].loc[0]

"Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)"

In [19]:
train_data[['dialogue', 'summary', 'summary_zh']].head()

,dialogue,summary,summary_zh
0,Amanda: I baked cookies. Do you want some? Jer...,Amanda baked cookies and will bring Jerry some...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up? Kim: Bad mood tbh, I was g...",Kim may try the pomodoro technique recommended...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something Sam: i d...,"Sam is confused, because he overheard Rick com...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [20]:
test_data[['dialogue', 'summary']].head()

,dialogue,summary
0,"Hannah: Hey, do you have Betty's number? Amand...",Hannah needs Betty's number but Amanda doesn't...
1,Eric: MACHINE! Rob: That's so gr8! Eric: I kno...,Eric and Rob are going to watch a stand-up on ...
2,"Lenny: Babe, can you help me with something? B...",Lenny can't decide which trousers to buy. Bob ...
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...
4,"Ollie: Hi , are you in Warsaw Jane: yes, just ...",Jane is in Warsaw. Ollie and Jane has a party....


In [21]:
val_data[['dialogue', 'summary']].head()

,dialogue,summary
0,"A: Hi Tom, are you busy tomorrow’s afternoon? ...",A will go to the animal shelter tomorrow to ge...
1,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,Jackie: Madison is pregnant Jackie: but she do...,Madison is pregnant but she doesn't want to ta...
3,Marla: <file_photo> Marla: look what I found u...,Marla found a pair of boxers under her bed.
4,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [22]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 1. Convert Pandas DataFrames to Hugging Face Dataset format
# Utilizing the previously created train_data, val_data, and test_data.
hg_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_data),
    "validation": Dataset.from_pandas(val_data),
    "test": Dataset.from_pandas(test_data)
})

# 2. Load the Model and Tokenizer
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# 3. Tokenization preprocessing function (applying max_length based on the paper)
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["dialogue"],
        max_length=1024,
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=150,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 3. Tokenization preprocessing function (applying max_length based on the paper)
def tokenize_function_translated(examples):
    model_inputs = tokenizer(
        examples["dialogue"],
        max_length=1024,
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["summary_zh"],
        max_length=150,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map tokenization to the entire dataset (batched=True for speed optimization)
print("start tokenizing")
tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)
translated_tokenized_datasets = hg_dataset.map(tokenize_function_translated, batched=True)
print("tokenization complete!")

# 4. Set up Data Collator (Dynamic padding within batches)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. M4 Pro customized training setup (Implementing the paper's batch size of 24)
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-dialogue-summary-final",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,       # Reduced from 8 to 2
    gradient_accumulation_steps=12,      # Increased to 12 (2 * 12 = 24, same as paper)
    per_device_eval_batch_size=2,        # Reduced for evaluation as well
    gradient_checkpointing=True,         # Crucial: Trades compute for memory
    num_train_epochs=20,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    fp16=False,
)

# 6. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Add validation set!
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Ready to do trainer.train()")

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


start tokenizing


Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

tokenization complete!
Ready to do trainer.train()


In [ ]:
# =====================================================================
# STEP 1: SELECT SMALL SUBSET FOR TESTING
# =====================================================================
# Take only the first 500 samples for a quick sanity check
small_train_dataset = tokenized_datasets["train"].select(range(500))
small_eval_dataset = tokenized_datasets["validation"].select(range(100))

# =====================================================================
# STEP 2: MODIFIED TRAINING ARGUMENTS FOR SPEED
# =====================================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-test-run",
    eval_strategy="no",
    learning_rate=2e-5,

    # Increase batch size slightly to speed up M4 Pro (since we have 48GB)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=6,       # 4 * 6 = 24 (Still effective batch size 24)

    # Use max_steps for a quick exit
    max_steps=50,                        # Stop training exactly at 50 steps
    num_train_epochs=1,                  # Ensure it doesn't run more than 1 epoch

    optim="adafactor",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    logging_steps=10,                    # Log every 10 steps to see progress
    save_strategy="no",                  # Don't waste time saving checkpoints during test
    predict_with_generate=True,
    fp16=False,
    bf16=False,
    report_to="none"
)

# =====================================================================
# STEP 3: INITIALIZE & RUN
# =====================================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,   # Use the sliced small dataset
    processing_class=tokenizer,
    data_collator=data_collator,
)

print(" Starting a high-speed test run (50 steps)...")
trainer.train()

In [23]:
pip install evaluate

In [24]:
pip install rouge_score

In [ ]:
# Load ROUGE metric
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    print("Inside compute metrics")
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode predictions and labels into text
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# Run evaluation on the test set
print(" Evaluating on the test set...")
results = trainer.evaluate(eval_dataset=translated_tokenized_datasets["test"], metric_key_prefix="test", compute_metrics=compute_metrics)
print(results)

In [10]:
# Initialize translation tokenizer and model: https://huggingface.co/Helsinki-NLP/opus-mt-en-zhhttps://huggingface.co/Helsinki-NLP/opus-mt-en-zh

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translate_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-zh")
translate_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-zh")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [25]:
def summarize_dialogue(text):
    # 1. Prepare input - this returns a dict containing 'input_ids' and 'attention_mask'
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(model.device)

    # 2. Generate summary - explicitly pass both input_ids and attention_mask
    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"], # Crucial for MPS compatibility
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # 3. Decode summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # 4. Translate summary
    tokenized_summary = translate_tokenizer(summary, return_tensors="pt").to(translate_model.device)
    output = translate_model.generate(**tokenized_summary)

    # 5. Decode and return translated summary
    decoded_output = translate_tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded_output

# Test with the sample dialogue again
sample_text = "Yun: Hey, did you finish the training? Yunmin: Yes, it's done! Yun: Great, what's next?"
print(f"Summary: {summarize_dialogue(sample_text)}")

Summary: Yun:嘿,你完成了训练吗?Yunmin:是的,结束了!很好,接下来是什么?


In [32]:
# Evaluation

# Install dependencies
!pip install pyonmttok jieba six nltk absl-py

# Install the xl-sum multilingual rouge_score fork
# This overwrites the standard rouge_score package with the multilingual version.
!pip install git+https://github.com/csebuetnlp/xl-sum.git#subdirectory=multilingual_rouge_scoring

# BERTScore
!pip install bert-score transformers

  Cloning https://github.com/csebuetnlp/xl-sum.git to /tmp/pip-req-build-g3dg29tf
  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/xl-sum.git /tmp/pip-req-build-g3dg29tf
  Resolved https://github.com/csebuetnlp/xl-sum.git to commit afcd803c8d7c98f3be60e9a7ce57fcbc3e729e8c
  Preparing metadata (setup.py) ... done


In [33]:
# Configuration

# ClidSum paper uses chinese-bert-wwm-ext for Chinese BERTScore.
# HuggingFace model ID:
BERTSCORE_MODEL = "hfl/chinese-bert-wwm-ext"

# ClidSum paper uses R1/R2/R-L
ROUGE_TYPES = ["rouge1", "rouge2", "rougeL"]

In [ ]:
references = dataset_dict['test']['summary_zh']
predictions = [summarize_dialogue(text) for text in dataset_dict['test']['dialogue']]

# Sanity checks
assert len(predictions) == len(references)

summary_eval_pair_scores = {
    'references':  references,
    'predictions': predictions,
    'rougescore':  [],   # one dict per pair: {'rouge1': float, 'rouge2': float, 'rougeL': float}
    'bertscore':   [],   # one F1 float per pair
}


In [61]:
# ROUGE Evaluation for Chinese
from rouge_score import rouge_scorer

def compute_rouge(
    predictions: list[str],
    references: list[str],
    rouge_types: list[str],
    language: str = "zh",
) -> tuple[dict, list[dict]]:

    scorer = rouge_scorer.RougeScorer(
        rouge_types=rouge_types,
        lang=language,
    )

    pair_scores = []
    aggregated = {rt: 0.0 for rt in rouge_types}

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)  # (reference, hypothesis)
        pair = {rt: round(scores[rt].fmeasure * 100, 2) for rt in rouge_types}
        pair_scores.append(pair)
        for rt in rouge_types:
            aggregated[rt] += scores[rt].fmeasure  # extract only F1 score

    n = len(predictions)
    corpus_scores = {rt: round(aggregated[rt] / n * 100, 2) for rt in rouge_types}

    return corpus_scores, pair_scores


rouge_corpus_scores, rouge_pair_scores = compute_rouge(predictions, references, ROUGE_TYPES, language="zh")

summary_eval_pair_scores['rougescore'] = rouge_pair_scores

print(f"  ROUGE-1 (R1) : {rouge_corpus_scores['rouge1']:.2f}")
print(f"  ROUGE-2 (R2) : {rouge_corpus_scores['rouge2']:.2f}")
print(f"  ROUGE-L (R-L): {rouge_corpus_scores['rougeL']:.2f}")


  ROUGE-1 (R1) : 20.89
  ROUGE-2 (R2) : 13.76
  ROUGE-L (R-L): 18.93


In [62]:
# BERTScore Evaluation for Chinese
from bert_score import BERTScorer

def compute_bertscore(
    predictions: list[str],
    references: list[str],
    model_type: str,
    lang: str = "zh",
    batch_size: int = 32,
    verbose: bool = True,
) -> tuple[dict, list]:

    scorer = BERTScorer(
        model_type=model_type,
        lang=lang,
        batch_size=batch_size,
        num_layers=8,  # matches bert-base-chinese optimal layer per BERTScore paper
    )

    # Fix the OverflowError
    scorer._tokenizer.model_max_length = 512

    P, R, F1 = scorer.score(
        predictions,
        references,
        verbose=verbose,
        batch_size=batch_size,
    )

    pair_scores = [round(F1[i].item() * 100, 2) for i in range(len(predictions))]
    corpus_scores = {"f1": round(F1.mean().item() * 100, 2)}

    return corpus_scores, pair_scores

bert_corpus_scores, bert_pair_scores = compute_bertscore(
    predictions,
    references,
    model_type=BERTSCORE_MODEL,
    lang="zh",
    batch_size=32,
)

summary_eval_pair_scores['bertscore'] = bert_pair_scores

print(f"  F1 (B-S)  : {bert_corpus_scores['f1']}")

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/hfl/chinese-bert-wwm-ext/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: hfl/chinese-bert-wwm-ext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.09 seconds, 2.76 sentences/sec
  F1 (B-S)  : 62.4


In [84]:
# Combine and Save Results

import pandas as pd

# combine corpus level scores
corpus_df = pd.DataFrame(rouge_corpus_scores | bert_corpus_scores, index=[0])
print(f'Corpus Eval Scores of {DATASET_NAME}')
display(corpus_df)

# combine individual pair scores
rouge_df = pd.DataFrame(summary_eval_results['rougescore'])  # cols: rouge1, rouge2, rougeL
bert_df  = pd.DataFrame({'bs_f1': summary_eval_results['bertscore']})

pair_results_df = pd.DataFrame({
    'reference':  summary_eval_results['references'],
    'prediction': summary_eval_results['predictions'],
} | rouge_df.to_dict('list') | bert_df.to_dict('list'))

# display analysis
print("\n── BERTScore F1 distribution ──")
print(pair_results_df['bs_f1'].describe().round(2))

TOP_N_WORST = 5
worst = pair_results_df.nsmallest(TOP_N_WORST, 'rougeL')
print(f'\n── Top {TOP_N_WORST} worst examples by ROUGE-L of {DATASET_NAME} ──')
display(worst)

# save results
corpus_df.to_csv(f"corpus_scores_{DATASET_NAME}.csv", index=False, encoding="utf-8-sig")
pair_results_df.to_csv(f"pair_scores_{DATASET_NAME}.csv", index=True, encoding="utf-8-sig")

Corpus Eval Scores of XSAMSum


,rouge1,rouge2,rougeL,f1
0,20.89,13.76,18.93,62.4



── BERTScore F1 distribution ──
count     3.00
mean     62.40
std       9.87
min      53.39
25%      57.12
50%      60.85
75%      66.90
max      72.95
Name: bs_f1, dtype: float64

── Top 5 worst examples by ROUGE-L of XSAMSum ──


,reference,prediction,rouge1,rouge2,rougeL,bs_f1
2,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,"Lenny: 贝比,你能帮我吗? 鲍勃: 当然,什么是 -== -= -= -= -= -=...",7.27,3.77,7.27,53.39
1,埃里克和罗伯要在youtube上看一场单口相声。,艾瑞克:MACHINE! 马希内!,8.33,0.00,8.33,60.85
0,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,"嘿,你有贝蒂的电话号码吗?",47.06,37.50,41.18,72.95
